# Batch layer trên Google Colab: CSE-CIC-IDS2018 với PySpark

Notebook chạy toàn bộ batch layer trên Colab: ETL và báo cáo chất lượng dữ liệu, kịch bản 1, RF đa lớp, kịch bản 2, benchmark mở rộng, biểu đồ.

| Thành phần | Vị trí | Ghi chú |
|---|---|---|
| Mã nguồn | `MyDrive/ie212-ids2018` | Clone từ GitHub, tự `git pull` ở mỗi phiên |
| Dữ liệu CSV | `/content/data` | Tải từ AWS Open Data, mất khi ngắt phiên |
| Parquet trung gian | `/content/work/parquet`, sao lưu ở `MyDrive/ie212_parquet` (~3,4 GB) | Phiên sau khôi phục từ Drive, không phải chạy lại ETL |
| Kết quả, mô hình | `MyDrive/ie212-ids2018/colab_out/{results,models}` | Tách riêng khỏi `results/` của bản chạy Docker để so sánh |

Colab miễn phí có 2 vCPU và khoảng 12 GB RAM, nên toàn bộ pipeline có thể cần nhiều phiên. Mọi bước đều chạy tiếp được: kết quả nằm trên Drive, kịch bản 2 có `--resume`. Khi bị ngắt phiên, chạy lại các ô **Thiết lập** rồi chạy tiếp từ bước đang dở.

## 0. Thiết lập (chạy lại ở mỗi phiên)

In [ ]:
# Tham số thực nghiệm (giữ giống bản Docker để so sánh được)
S1_TRAIN_FRAC = 0.3     # kịch bản 1 và đa lớp
S2_TRAIN_FRAC = 0.2     # kịch bản 2
BENCH_CORES   = '1,2'   # Colab chỉ có 2 vCPU
BENCH_FRACS   = '0.25,0.5,1.0'

from google.colab import drive
drive.mount('/content/drive')

import os
REPO_URL = 'https://github.com/phonguytin/ie212-ids2018.git'
PROJECT  = '/content/drive/MyDrive/ie212-ids2018'
OUT      = f'{PROJECT}/colab_out'
RESULTS, MODELS = f'{OUT}/results', f'{OUT}/models'
PARQUET_BACKUP  = '/content/drive/MyDrive/ie212_parquet'
WORK, DATA      = '/content/work', '/content/data'

if not os.path.exists(f'{PROJECT}/.git'):
    !git clone -q {REPO_URL} "{PROJECT}"
else:
    !git -C "{PROJECT}" pull -q --ff-only
for d in (RESULTS, MODELS, WORK, DATA):
    os.makedirs(d, exist_ok=True)
!git -C "{PROJECT}" log --oneline -1

In [ ]:
# Java 17 + PySpark 4.0.1: cùng phiên bản với image Docker để mô hình lưu trên Colab nạp được ở speed layer
!apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
!pip -q install pyspark==4.0.1 pyarrow scikit-learn psutil matplotlib awscli
import sys
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PYTHONPATH'] = f'{PROJECT}/src'
os.environ['SPARK_DRIVER_MEMORY'] = '9g'
sys.path.insert(0, f'{PROJECT}/src')

def n_csv(d):
    return len([f for f in os.listdir(d) if f.endswith('.csv')]) if os.path.isdir(d) else 0

def ensure_data():
    """Tải 10 file CSV (6,9 GB) về đĩa Colab nếu chưa có. Chỉ cần cho ETL và benchmark."""
    if n_csv(DATA) < 10:
        !aws s3 cp --no-sign-request --only-show-errors "s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms/" {DATA} --recursive
    print(n_csv(DATA), 'file CSV trong', DATA)

def ensure_parquet():
    """Khôi phục Parquet từ Drive nếu phiên mới. Trả về True nếu đã sẵn sàng."""
    if os.path.exists(f'{WORK}/parquet/meta.json'):
        return True
    if os.path.exists(f'{PARQUET_BACKUP}/meta.json'):
        print('Khôi phục Parquet từ Drive...')
        !cp -r "{PARQUET_BACKUP}" {WORK}/parquet
        return True
    return False

!java -version 2>&1 | head -1
!python --version; nproc; free -g | head -2

## 1. ETL và báo cáo chất lượng dữ liệu

Đọc 10 file CSV (~16 triệu luồng), làm sạch, khử trùng lặp, ghi Parquet. Parquet được sao lưu lên Drive, nên ô này chỉ cần chạy **một lần**.

In [ ]:
%%time
if ensure_parquet():
    print('Parquet đã sẵn sàng, bỏ qua ETL')
else:
    ensure_data()
    !cd "{PROJECT}" && python -m ids.etl --input {DATA} --output {WORK}/parquet --report "{RESULTS}/quality_report.json"
    !rm -rf "{PARQUET_BACKUP}" && cp -r {WORK}/parquet "{PARQUET_BACKUP}"
!du -sh {WORK}/parquet/clean {WORK}/parquet/dedup

## 2. Kịch bản 1: chia ngẫu nhiên, có trùng lặp và đã khử trùng lặp

Huấn luyện LR, DT, RF, GBT trên hai biến thể dữ liệu; quét ngưỡng chi phí trên validation; chọn mô hình có chi phí nhỏ nhất ở tỷ lệ 1:50 và lưu vào `colab_out/models`.

In [ ]:
%%time
assert ensure_parquet(), 'Chưa có Parquet: chạy bước 1 trước'
!cd "{PROJECT}" && python -m ids.train scenario1 --parquet {WORK}/parquet --work {WORK} --results "{RESULTS}" --models-out "{MODELS}" --preset full --train-frac {S1_TRAIN_FRAC}

## 3. Random Forest đa lớp (7 họ tấn công)

In [ ]:
%%time
assert ensure_parquet(), 'Chưa có Parquet: chạy bước 1 trước'
!cd "{PROJECT}" && python -m ids.train multiclass --parquet {WORK}/parquet --work {WORK} --results "{RESULTS}" --models-out "{MODELS}" --preset full --train-frac {S1_TRAIN_FRAC}

## 4. Kịch bản 2: leave-one-day-out

40 lần huấn luyện (10 ngày × 4 mô hình). Kết quả được ghi sau mỗi ngày; nếu bị ngắt phiên, chạy lại ô này và các ngày đã xong sẽ được bỏ qua (`--resume`).

In [ ]:
%%time
assert ensure_parquet(), 'Chưa có Parquet: chạy bước 1 trước'
!cd "{PROJECT}" && python -m ids.train scenario2 --parquet {WORK}/parquet --work {WORK} --results "{RESULTS}" --models-out "{MODELS}" --preset fast --train-frac {S2_TRAIN_FRAC} --resume

## 5. Benchmark khả năng mở rộng: Spark và pandas/scikit-learn

Mẫu dữ liệu thật 25% ⊂ 50% ⊂ 100%. Mỗi lần đo chạy trong tiến trình riêng, ghi thời gian, RAM đỉnh và trạng thái (`ok`, `oom`, `timeout`). Kết quả đo được ghi dần nên có thể chạy tiếp bằng `--resume`.

In [ ]:
%%time
assert ensure_parquet(), 'Chưa có Parquet: chạy bước 1 trước'
ensure_data()
!cd "{PROJECT}" && python -m ids.bench prepare --data {DATA} --parquet {WORK}/parquet --bench {WORK}/bench
!cd "{PROJECT}" && python -m ids.bench all --data {DATA} --parquet {WORK}/parquet --bench {WORK}/bench --cores {BENCH_CORES} --fracs {BENCH_FRACS} --timeout 3600 --spark-mem 8g --note "Google Colab" --out "{RESULTS}/scalability.json" --resume

## 6. Biểu đồ và bảng tổng hợp

In [ ]:
!cd "{PROJECT}" && python -m ids.figures --results "{RESULTS}"
from IPython.display import Image, Markdown, display
display(Markdown(open(f'{RESULTS}/summary.md', encoding='utf-8').read()))
for f in sorted(os.listdir(f'{RESULTS}/figures')):
    print(f); display(Image(f'{RESULTS}/figures/{f}'))

## 7. Bàn giao kết quả

1. Tải thư mục `MyDrive/ie212-ids2018/colab_out/` về máy, đặt vào `colab_out/` trong repo.
2. Ghi lại thời gian chạy (`Wall time`) và cấu hình máy (`nproc`, `free -g`) của từng bước.
3. Tạo nhánh mới, commit `colab_out/` (không commit Parquet), mở Pull Request.